# Deploying a Fine-Tuned LLM with FastAPI and Docker

In this notebook, we'll walk through the process of deploying a LoRA fine-tuned model using FastAPI and Docker. This approach provides a scalable and maintainable way to serve your model for inference.

## 1. Install Required Dependencies

In [ ]:
!pip install fastapi uvicorn transformers torch peft accelerate bitsandbytes

## 2. Create a FastAPI Application

First, let's create a FastAPI application that will serve our fine-tuned model. We'll define endpoints for text generation and model information.

In [ ]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
import time

app = FastAPI(title="Fine-Tuned LLM API", 
              description="API for serving a LoRA fine-tuned language model",
              version="1.0.0")

# Model configuration
MODEL_ID = "meta-llama/Llama-2-7b-hf"  # Replace with your model or Hugging Face model ID
ADAPTER_ID = "./lora-adapter"  # Path to your LoRA adapter (if using PEFT)
USE_PEFT = False  # Set to True if using a LoRA adapter
USE_8BIT = True  # Set to True for 8-bit quantization

# Global variables for model and tokenizer
model = None
tokenizer = None

# Request and response models
class GenerationRequest(BaseModel):
    prompt: str
    max_length: int = 100
    temperature: float = 0.7
    top_p: float = 0.9
    do_sample: bool = True

class GenerationResponse(BaseModel):
    generated_text: str
    generation_time: float

class ModelInfoResponse(BaseModel):
    model_id: str
    is_peft: bool
    is_quantized: bool
    device: str

@app.on_event("startup")
async def startup_event():
    """Load model and tokenizer on startup"""
    global model, tokenizer
    
    print(f"Loading tokenizer from {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    
    print(f"Loading model from {MODEL_ID}...")
    if USE_8BIT:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto",
            load_in_8bit=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto"
        )
    
    if USE_PEFT and os.path.exists(ADAPTER_ID):
        from peft import PeftModel
        print(f"Loading LoRA adapter from {ADAPTER_ID}...")
        model = PeftModel.from_pretrained(model, ADAPTER_ID)
    
    print("Model loaded successfully!")

@app.get("/", response_model=dict)
async def root():
    """Root endpoint with API information"""
    return {
        "message": "Fine-Tuned LLM API",
        "docs": "/docs",
        "endpoints": ["/generate", "/model-info"]
    }

@app.get("/model-info", response_model=ModelInfoResponse)
async def get_model_info():
    """Get information about the loaded model"""
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")
    
    return {
        "model_id": MODEL_ID,
        "is_peft": USE_PEFT,
        "is_quantized": USE_8BIT,
        "device": str(model.device)
    }

@app.post("/generate", response_model=GenerationResponse)
async def generate_text(request: GenerationRequest):
    """Generate text based on the provided prompt"""
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")
    
    try:
        # Tokenize input
        inputs = tokenizer(request.prompt, return_tensors="pt").to(model.device)
        
        # Start timing
        start_time = time.time()
        
        # Generate text
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=request.max_length,
            do_sample=request.do_sample,
            temperature=request.temperature,
            top_p=request.top_p
        )
        
        # End timing
        generation_time = time.time() - start_time
        
        # Decode output
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        return {
            "generated_text": generated_text,
            "generation_time": generation_time
        }
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

## 3. Create a Dockerfile

Next, let's create a Dockerfile to containerize our application:

In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \
    build-essential \
    git \
    && rm -rf /var/lib/apt/lists/*

# Copy requirements file
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app.py .

# Copy model files (if using local models)
# COPY lora-adapter/ ./lora-adapter/

# Expose the port
EXPOSE 8000

# Run the API server
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

## 4. Create a requirements.txt file

In [ ]:
%%writefile requirements.txt
fastapi>=0.95.0
uvicorn>=0.22.0
transformers>=4.30.0
torch>=2.0.0
peft>=0.4.0
accelerate>=0.20.0
bitsandbytes>=0.40.0
pydantic>=2.0.0

## 5. Build and Run the Docker Container

Now, let's build and run the Docker container:

In [ ]:
# Build the Docker image
!docker build -t llm-api .

In [ ]:
# Run the Docker container
!docker run -d -p 8000:8000 --name llm-api-container llm-api

## 6. Test the API

Let's test our API by sending requests to it:

In [ ]:
import requests
import json

# Test the model info endpoint
response = requests.get("http://localhost:8000/model-info")
print("Model Info:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# Test the generate endpoint
data = {
    "prompt": "The best way to learn about LLM fine-tuning is",
    "max_length": 100,
    "temperature": 0.7
}

response = requests.post("http://localhost:8000/generate", json=data)
result = response.json()

print(f"Generated Text (in {result['generation_time']:.2f} seconds):")
print(result["generated_text"])

## 7. Clean Up

When you're done, stop and remove the Docker container:

In [ ]:
# Stop the container
!docker stop llm-api-container

# Remove the container
!docker rm llm-api-container

## 8. Deployment to Cloud Platforms

For production deployment, you can push your Docker image to a container registry and deploy it to a cloud platform:

### AWS Deployment

```bash
# Tag the image for ECR
docker tag llm-api:latest <your-aws-account-id>.dkr.ecr.<region>.amazonaws.com/llm-api:latest

# Push to ECR
docker push <your-aws-account-id>.dkr.ecr.<region>.amazonaws.com/llm-api:latest
```

Then deploy using ECS, EKS, or SageMaker.

### Google Cloud Platform

```bash
# Tag the image for GCR
docker tag llm-api:latest gcr.io/<your-project-id>/llm-api:latest

# Push to GCR
docker push gcr.io/<your-project-id>/llm-api:latest
```

Then deploy using Cloud Run, GKE, or Vertex AI.

## 9. Conclusion

In this notebook, we've demonstrated how to:

1. Create a FastAPI application to serve a fine-tuned LLM
2. Containerize the application with Docker
3. Build and run the Docker container
4. Test the API endpoints
5. Clean up resources
6. Deploy to cloud platforms (instructions)

This approach provides a scalable and maintainable way to deploy your fine-tuned models for inference in production environments.